# Taylor Root Prediction — Reviewer Demo Notebook (with Dataset Generation)

이 노트북은 리뷰어가 **GitHub 레포를 clone 한 뒤**, 다음을 재현할 수 있도록 구성했습니다.

1) (선택) **데이터셋 생성**(NPZ 생성)  
2) (선택) 모델 학습(ANN/LSTM/MLP/Transformer)  
3) 평가(evaluate_k_sweep) + 그림/리포트 생성  

> ✅ 모든 경로는 **레포 루트 기준 상대경로**로 처리됩니다.  
> 실행 위치가 어디든 repo-root 자동 탐색으로 경로가 깨지지 않게 합니다.


In [1]:
# (선택) requirements 설치
# !pip install -r requirements.txt

# 또는 최소 설치(필수 + 선택)
# !pip install numpy torch tqdm pyyaml matplotlib pillow sympy requests


## 1) Repo Root 자동 탐색 + 실행 유틸


In [2]:
from __future__ import annotations
from pathlib import Path
import os, sys, subprocess, re
import numpy as np

def find_repo_root(start: Path | None = None) -> Path:
    if start is None:
        start = Path.cwd().resolve()
    else:
        start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "configs").is_dir() and ((p / "README.md").exists() or (p / "LICENSE").exists()):
            return p
    for p in [start] + list(start.parents):
        if (p / "configs").is_dir():
            return p
    return start

REPO = find_repo_root()
print("REPO_ROOT =", REPO)

def R(rel: str | os.PathLike | None) -> Path | None:
    if rel is None:
        return None
    s = str(rel).strip()
    if s == "":
        return None
    s = os.path.expanduser(os.path.expandvars(s))
    p = Path(s)
    if p.is_absolute():
        return p
    return (REPO / p).resolve()

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def run(cmd, env=None, cwd=None):
    print(">>", " ".join(map(str, cmd)))
    r = subprocess.run(cmd, env=env, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if r.stdout:
        print("----- STDOUT -----")
        print(r.stdout)
    if r.stderr:
        print("----- STDERR -----")
        print(r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Command failed (code={r.returncode})")
    return r


REPO_ROOT = /home/seokjun/taylor-root-prediction


## 2) 레포 구조 점검
- configs 존재 확인
- data 폴더 내 npz 존재 확인(없으면 아래 3)에서 생성)


In [3]:
cfgs = {
    "ann": R("configs/taylor_root_ann.yaml"),
    "lstm": R("configs/taylor_root_lstm.yaml"),
    "mlp": R("configs/taylor_root_mlp.yaml"),
    "transformer": R("configs/transformer_interval.yaml"),
    "eval": R("configs/eval_k_sweep.yaml"),
}
for k, p in cfgs.items():
    print(f"[{k:12s}] {p}  exists={p.exists() if p else False}")

print("\n[DATA DIR SCAN] npz under repo/data (show up to 50):")
data_root = R("data")
if data_root and data_root.exists():
    npzs = sorted(list(data_root.rglob("*.npz")))
    for p in npzs[:50]:
        print(" ", p.relative_to(REPO))
    print("  ... total npz:", len(npzs))
else:
    print("  data/ does not exist yet. (OK)")


[ann         ] /home/seokjun/taylor-root-prediction/configs/taylor_root_ann.yaml  exists=True
[lstm        ] /home/seokjun/taylor-root-prediction/configs/taylor_root_lstm.yaml  exists=True
[mlp         ] /home/seokjun/taylor-root-prediction/configs/taylor_root_mlp.yaml  exists=True
[transformer ] /home/seokjun/taylor-root-prediction/configs/transformer_interval.yaml  exists=True
[eval        ] /home/seokjun/taylor-root-prediction/configs/eval_k_sweep.yaml  exists=True

[DATA DIR SCAN] npz under repo/data (show up to 50):
  ... total npz: 0


## 3) (선택) 데이터셋 생성 (NPZ)

이 레포의 학습/평가가 돌아가려면 다음 형태의 데이터가 필요합니다.

- Root regression (ANN/LSTM/MLP):  
  `data/.../taylor_deg25_{train,val,test}.npz` (keys: coeffs, root0, root1, root2, ...)
- Transformer interval:  
  `data/.../taylor_deg25_{train,val,test}.npz` (keys: expr_str(or func_expr), roots, ...)

아래 셀은:
- data 폴더에서 필요한 npz가 없으면
- 레포 안에서 “데이터 생성 스크립트”를 자동 탐색해서 실행합니다.

> ⚠️ 큰 dataset_size(예: 1,000,000)는 컴퓨팅/시간/디스크가 많이 필요할 수 있습니다.  
> 리뷰어용으로는 먼저 작은 N으로 생성/학습/평가가 동작하는지 확인한 뒤 늘리는 것을 권장합니다.


In [ ]:
# ---- 리뷰어용 기본값(필요하면 수정) ----
DEGREE = 25

# 빠른 재현용(작게) 예시: 20000~100000 정도 추천(환경에 따라 조절)
N_TOTAL_ROOT = 20000
N_TOTAL_INTERVAL = 20000

# 생성 결과 저장 위치(상대경로)
OUT_ROOT_DIR = R("taylor_data_physchem_v4_deg25")
OUT_INTERVAL_DIR = R("data/taylor_data_physchem_v4_interval")

ensure_dir(OUT_ROOT_DIR)
ensure_dir(OUT_INTERVAL_DIR)

# 필요한 파일 경로
train_npz = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_train.npz"
val_npz   = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_val.npz"
test_npz  = OUT_ROOT_DIR / f"taylor_deg{DEGREE}_test.npz"

interval_train_npz = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_train.npz"
interval_val_npz   = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_val.npz"
interval_test_npz  = OUT_INTERVAL_DIR / f"taylor_deg{DEGREE}_test.npz"

print("[TARGET ROOT DATA]")
for p in [train_npz, val_npz, test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())
print("\n[TARGET INTERVAL DATA]")
for p in [interval_train_npz, interval_val_npz, interval_test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())

# ---- 데이터 생성 스크립트 자동 탐색 ----
def find_generator(pattern_keywords):
    py_files = list(REPO.rglob("*.py"))
    cand = []
    for p in py_files:
        name = p.name.lower()
        if any(k in name for k in pattern_keywords):
            cand.append(p)
    # 1차: 파일명 힌트
    if cand:
        return sorted(cand)[0]

    # 2차: 내용 검색(가볍게 상위 몇천 개만)
    for p in py_files[:4000]:
        try:
            txt = p.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            continue
        if all(k in txt for k in pattern_keywords):
            return p
    return None

# root-regression generator: 너가 준 코드 특징 문자열을 키워드로 탐색
gen_root = find_generator(["generate_dataset", "templates.json"])
# interval generator: roots + expr_str 저장 형태를 키워드로 탐색
gen_interval = find_generator(["expr_str", "roots", "taylor_deg25"])

print("\n[FOUND GENERATORS]")
print(" root_gen     =", gen_root.relative_to(REPO) if gen_root else None)
print(" interval_gen =", gen_interval.relative_to(REPO) if gen_interval else None)

if gen_root is None:
    print("\n[WARN] root dataset generator script not found automatically.")
    print("      Put the generator under e.g. data_generation/ and re-run this cell.")
if gen_interval is None:
    print("\n[WARN] interval dataset generator script not found automatically.")
    print("      Put the interval generator under e.g. data_generation/ and re-run this cell.")


[TARGET ROOT DATA]
  data/taylor_data_physchem_v4_deg25/taylor_deg25_train.npz exists= False
  data/taylor_data_physchem_v4_deg25/taylor_deg25_val.npz exists= False
  data/taylor_data_physchem_v4_deg25/taylor_deg25_test.npz exists= False

[TARGET INTERVAL DATA]
  data/taylor_data_physchem_v4_interval/taylor_deg25_train.npz exists= False
  data/taylor_data_physchem_v4_interval/taylor_deg25_val.npz exists= False
  data/taylor_data_physchem_v4_interval/taylor_deg25_test.npz exists= False

[FOUND GENERATORS]
 root_gen     = scripts/data/generate_dataset_physchem_v4.py
 interval_gen = models/transformer/model.py


### 3-A) Root regression 데이터 생성 실행(없을 때만)

- 생성 스크립트가 argparse를 쓴다고 가정하고 `--out-dir`, `--degree`, `--n-total` 등을 전달합니다.  
- 만약 네 스크립트 옵션명이 다르면, 아래 `cmd = [...]` 부분만 바꿔주면 됩니다.


In [5]:
# root regression dataset 생성(필요할 때만)
need_root = (not train_npz.exists()) or (not val_npz.exists()) or (not test_npz.exists())

if not need_root:
    print("[SKIP] root npz already exists.")
else:
    assert gen_root is not None, "Generator script not found. Place it inside repo and re-run previous cell."
    cmd = [
        sys.executable, str(gen_root),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_ROOT),
        "--seed", "42",
        "--out-dir", str(OUT_ROOT_DIR.relative_to(REPO)),
        "--save-expr-str", "1",
    ]
    # note: 옵션명이 다르면 여기 수정
    run(cmd, env={"PYTHONPATH": str(REPO), **os.environ}, cwd=str(REPO))

print("[ROOT DATA CHECK]")
for p in [train_npz, val_npz, test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


>> /home/seokjun/miniconda3/envs/action_recognition/bin/python /home/seokjun/taylor-root-prediction/scripts/data/generate_dataset_physchem_v4.py --degree 25 --n-total 20000 --seed 42 --out-dir data/taylor_data_physchem_v4_deg25 --save-expr-str 1
----- STDOUT -----
[INFO] cfg_path=configs/dataset_physchem_v4_deg25.yaml
[INFO] degree=25, n_total=1000000, seed=42
[INFO] out_dir=./taylor_data_physchem_v4_deg25
[INFO] n_templates=5000, max_terms=3, max_depth=2
[INFO] root_range=1.0 → [-1.0, 1.0]
[INFO] TRANSC_TERMS=['sin', 'cos', 'exp', 'sinh', 'cosh', 'tanh', 'ln', 'log', 'inv', 'sqrt1p']
[INFO] poly_degs=[1, 2, 3, 4, 5, 7, 10], min_poly_norm=1e-08
[INFO] (filter) y_abs_max=50.0, y_ptp_min=0.001, y_grid_n=201
[INFO] max_roots_keep=8
[INFO] save_expr_str=True, float_sig=6
[INFO] max_retry_factor=80
[INFO] 각 샘플은 최종적으로 max |coeff| = 1 로 정규화됩니다.

[INFO] #atomic term exprs (<= depth 2) = 198
[INFO] Generated 5000 templates (target=5000, max_terms=3, max_depth=2)
[SAVE] templates list -> taylor_

### 3-B) Transformer interval 데이터 생성 실행(없을 때만)

Interval용 generator도 동일하게 실행합니다.


In [ ]:
need_interval = (not interval_train_npz.exists()) or (not interval_val_npz.exists()) or (not interval_test_npz.exists())

if not need_interval:
    print("[SKIP] interval npz already exists.")
else:
    assert gen_interval is not None, "Interval generator script not found. Place it inside repo and re-run previous cell."
    cmd = [
        sys.executable, str(gen_interval),
        "--degree", str(DEGREE),
        "--n-total", str(N_TOTAL_INTERVAL),
        "--seed", "42",
        "--out-dir", str(OUT_INTERVAL_DIR.relative_to(REPO)),
        "--save-expr-str", "1",
    ]
    # note: 옵션명이 다르면 여기 수정
    run(cmd, env={"PYTHONPATH": str(REPO), **os.environ}, cwd=str(REPO))

print("[INTERVAL DATA CHECK]")
for p in [interval_train_npz, interval_val_npz, interval_test_npz]:
    print(" ", p.relative_to(REPO), "exists=", p.exists())


>> /home/seokjun/miniconda3/envs/action_recognition/bin/python /home/seokjun/taylor-root-prediction/models/transformer/model.py --degree 25 --n-total 20000 --seed 42 --out-dir data/taylor_data_physchem_v4_interval --save-expr-str 1
----- STDERR -----
Traceback (most recent call last):
  File "/home/seokjun/taylor-root-prediction/models/transformer/model.py", line 769, in <module>
    main()
  File "/home/seokjun/taylor-root-prediction/models/transformer/model.py", line 748, in main
    train_from_yaml(
  File "/home/seokjun/taylor-root-prediction/models/transformer/model.py", line 544, in train_from_yaml
    expr_tr, roots_tr = load_npz_expr_roots(train_npz)
  File "/home/seokjun/taylor-root-prediction/models/transformer/model.py", line 457, in load_npz_expr_roots
    raise FileNotFoundError(f"Missing file: {p}")
FileNotFoundError: Missing file: /home/seokjun/taylor-root-prediction/data/taylor_data_physchem_v4_interval/taylor_deg25_train.npz



RuntimeError: Command failed (code=1)

## 4) (선택) 모델 학습 실행

체크포인트가 없다면 학습을 수행합니다.  
환경변수로 경로를 넘겨서 `src` import 문제도 같이 해결합니다(PYTHONPATH=REPO).


In [ ]:
import torch

def train_script(script_rel: str, cfg_rel: str, out_rel: str,
                 train_path: Path, val_path: Path, test_path: Path | None,
                 extra_env: dict | None = None):
    script = R(script_rel)
    assert script and script.exists(), f"Script not found: {script}"
    env = os.environ.copy()
    env.update({
        "TAYLOR_CFG": str(R(cfg_rel)),
        "TRAIN_NPZ": str(train_path),
        "VAL_NPZ": str(val_path),
        "TEST_NPZ": (str(test_path) if test_path is not None else ""),
        "OUT_DIR": str(R(out_rel)),
        "DEVICE": "cuda" if torch.cuda.is_available() else "cpu",
        "PYTHONPATH": str(REPO),
    })
    if extra_env:
        env.update(extra_env)
    run([sys.executable, str(script)], env=env, cwd=str(REPO))

# ---- 실제 실행 예시 (필요한 것만 주석 해제) ----
# train_script("models/taylor_nn/ann.py", "configs/taylor_root_ann.yaml", "results/taylor_nn/ann",
#              train_npz, val_npz, test_npz)

# train_script("models/taylor_nn/lstm.py", "configs/taylor_root_lstm.yaml", "results/taylor_nn/lstm",
#              train_npz, val_npz, test_npz)

# train_script("models/taylor_nn/mlp.py", "configs/taylor_root_mlp.yaml", "results/taylor_nn/mlp",
#              train_npz, val_npz, test_npz)

# train_script("models/transformer/model.py", "configs/transformer_interval.yaml", "results/transformer_interval",
#              interval_train_npz, interval_val_npz, interval_test_npz, extra_env={"MODE":"train"})


## 5) 평가 실행 (K sweep)


In [ ]:
eval_script = R("evaluation/evaluate_k_sweep.py")
assert eval_script and eval_script.exists(), f"Missing eval script: {eval_script}"

env = os.environ.copy()
env["EVAL_CFG"] = str(R("configs/eval_k_sweep.yaml"))
env["OUTDIR"]   = str(R("results/runs_k_sweep_viz"))
env["DEVICE"]   = "cuda" if torch.cuda.is_available() else "cpu"
env["PYTHONPATH"] = str(REPO)

run([sys.executable, str(eval_script)], env=env, cwd=str(REPO))


In [ ]:
from pathlib import Path
import json, glob
import pandas as pd

outdir = R("results/runs_k_sweep_viz")  # 노트북에서 evaluate outdir과 맞춰줘

# fail report json 찾기 (evaluate가 --report-fail-funcid --report-fail-save 켜져 있으면 생성됨)
cands = sorted(glob.glob(str(outdir / "fail_by_funcid_*.json")))
print("found:", len(cands))
for p in cands[:10]:
    print(" ", Path(p).name)

if not cands:
    raise RuntimeError("fail_by_funcid_*.json not found. evaluate yaml에서 reports.report_fail_funcid=True, report_fail_save=True로 켜줘.")

# 가장 최신/가장 관련 파일 하나 선택(필요하면 직접 지정)
p = Path(cands[-1])
rows = json.loads(p.read_text(encoding="utf-8"))

df = pd.DataFrame([{
    "func_id": r["func_id"],
    "name": r.get("name",""),
    "n_fail": r["n_fail"],
    "n_all": r["n_all"],
    "fail_rate(%)": 100.0 * r["fail_rate"],
    "top_reasons": "; ".join([f"{k}:{v}" for k,v in r.get("top_reasons", [])]),
    "expr_ex1": (r.get("expr_examples",[""])[0] if r.get("expr_examples") else ""),
} for r in rows])

df = df.sort_values(["n_fail","fail_rate(%)"], ascending=False)
display(df.head(30))

print("\nSummary:")
print(" total funcs:", df.shape[0])
print(" total fails:", int(df["n_fail"].sum()))
print(" worst fail_rate top5:")
display(df.sort_values("fail_rate(%)", ascending=False).head(5)[["func_id","name","n_fail","n_all","fail_rate(%)"]])


## 6) 결과 확인 (png/csv/json 탐색 + 일부 미리보기)


In [ ]:
from glob import glob
from pathlib import Path

outdir = R("results/runs_k_sweep_viz")
print("OUTDIR =", outdir)

pngs = sorted(glob(str(outdir / "**/*.png"), recursive=True))
csvs = sorted(glob(str(outdir / "**/*.csv"), recursive=True))
jsons = sorted(glob(str(outdir / "**/*.json"), recursive=True))

print(f"found png={len(pngs)}, csv={len(csvs)}, json={len(jsons)}")

# (옵션) png 미리보기
try:
    from PIL import Image
    import matplotlib.pyplot as plt

    for p in pngs[:6]:
        print("PNG:", Path(p).relative_to(REPO))
        img = Image.open(p)
        plt.figure(figsize=(10,4))
        plt.imshow(img)
        plt.axis('off')
        plt.title(Path(p).name)
        plt.show()
except Exception as e:
    print("Preview skipped:", type(e).__name__, e)

if csvs:
    print("\nCSV examples:")
    for p in csvs[:5]:
        print(" ", Path(p).relative_to(REPO))
if jsons:
    print("\nJSON examples:")
    for p in jsons[:5]:
        print(" ", Path(p).relative_to(REPO))


## 7) (옵션) 단일 샘플 sanity check


In [ ]:
sample_npz = test_npz
if sample_npz is None or (not sample_npz.exists()):
    print("test_npz not found. skip.")
else:
    data = np.load(sample_npz, allow_pickle=True)
    print("NPZ keys (head):", list(data.keys())[:20])
    coeffs = data["coeffs"].astype(np.float64)
    r0 = data["root0"].reshape(-1).astype(np.float64) if "root0" in data else None
    i = 0
    c = coeffs[i]
    print("coeffs shape:", coeffs.shape, "degree=", coeffs.shape[1]-1)
    if r0 is not None:
        print("root0[0] =", r0[i])

    def poly_eval_asc(coeffs_asc, x):
        p = 0.0
        for a in coeffs_asc[::-1]:
            p = p * x + float(a)
        return p

    xs = np.linspace(-1, 1, 401)
    ys = np.array([poly_eval_asc(c, x) for x in xs])
    print("P(x) abs max:", float(np.max(np.abs(ys[np.isfinite(ys)]))))
